# EdNet KT1 — Pipeline Huấn luyện XGBoost & SHAP (Bản Chuẩn Hóa Không Rò Rỉ Dữ Liệu)

Các cải tiến khoa học dữ liệu trong phiên bản này:
1. **Chống rò rỉ dữ liệu theo người học (User-level Group Split)**: Chia Train / Val / Test (70% / 15% / 15%) bằng `GroupShuffleSplit` theo `user_id` trước khi tính các thống kê toàn cục.
2. **Chống Target Leakage cho `question_difficulty`**: Chỉ tính độ khó câu hỏi trên tập **Train**, sau đó ánh xạ sang Val/Test (câu hỏi mới gán giá trị trung bình của tập Train).
3. **Chuẩn hóa đúng chiều biến `question_difficulty`**: Định nghĩa `question_difficulty = 1 - mean(is_correct)` (0.0 = Rất dễ, 1.0 = Rất khó), giúp giải thích SHAP nhất quán với trực giác sư phạm.
4. **Tách 2 mô hình chuyên biệt**:
   - `ednet_xgboost_model_final.json` (7 đặc trưng): Dùng cho **Bệnh án học tập (Post-Mortem)** và **Sổ tay Tiên tri** sau khi có `elapsed_time`.
   - `ednet_xgboost_prequiz_model.json` (6 đặc trưng, bỏ `elapsed_time`): Dùng cho **Thử thách XAI trước khi làm bài (Pre-Quiz)** để loại bỏ lỗi Out-of-Distribution (`elapsed_time = 0`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q -o "/content/drive/MyDrive/DoAn4/KT1-1000us.zip" -d "/content/kt1_data"
!pip install xgboost==2.0.3 shap scikit-learn==1.5.2 -q
print("✅ Đã giải nén dữ liệu và cài thư viện thành công!")


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss, roc_curve

# 1. Đọc toàn bộ CSV của 1000 học viên EdNet KT1
csv_files = glob.glob('/content/kt1_data/**/*.csv', recursive=True)
print(f"🔍 Tìm thấy {len(csv_files)} file users.")

df_list = []
for file in csv_files:
    filename = os.path.basename(file)
    user_id = filename.replace('u', '').replace('.csv', '')
    if user_id.isdigit():
        df_u = pd.read_csv(file)
        df_u['user_id'] = int(user_id)
        df_list.append(df_u)

full_df = pd.concat(df_list, ignore_index=True)
full_df = full_df.sort_values(by=['user_id', 'timestamp']).reset_index(drop=True)

# 2. Ghép đáp án đúng từ questions.csv
questions_df = pd.read_csv('/content/drive/MyDrive/DoAn4/questions.csv')
q_col = 'item_id' if 'item_id' in full_df.columns else 'question_id'

full_df = full_df.merge(
    questions_df[['question_id', 'correct_answer']],
    left_on=q_col,
    right_on='question_id',
    how='left'
)

full_df['user_answer'] = full_df['user_answer'].astype(str).str.strip().str.lower()
full_df['correct_answer'] = full_df['correct_answer'].astype(str).str.strip().str.lower()
full_df['is_correct'] = (full_df['user_answer'] == full_df['correct_answer']).astype(int)

# 3. Tạo các đặc trưng lịch sử nhân quả (strictly causal: dùng shift(1) trong từng user)
print("⚡ Đang tạo đặc trưng lịch sử hành vi (không rò rỉ tương lai)...")
full_df['user_history_acc'] = full_df.groupby('user_id')['is_correct'].transform(
    lambda x: x.shift(1).expanding().mean()
).fillna(0.5)

full_df['user_questions_attempted'] = full_df.groupby('user_id').cumcount()

full_df['rolling_acc_5'] = full_df.groupby('user_id')['is_correct'].transform(
    lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
).fillna(0.5)

full_df['time_diff'] = full_df.groupby('user_id')['timestamp'].diff().fillna(0) / 1000.0
full_df['log_time_diff'] = np.log1p(full_df['time_diff'].clip(lower=0))

if 'elapsed_time' in full_df.columns:
    # Cắt ngưỡng outlier cực đoan (> 5 phút = 300,000 ms) để ổn định phân phối
    full_df['elapsed_time'] = full_df['elapsed_time'].clip(lower=0, upper=300000)
    full_df['user_avg_elapsed_time'] = full_df.groupby('user_id')['elapsed_time'].transform(
        lambda x: x.shift(1).expanding().mean()
    )
else:
    full_df['elapsed_time'] = np.nan
    full_df['user_avg_elapsed_time'] = np.nan

print(f"✅ Tổng số lượt tương tác: {len(full_df):,} | Số học viên: {full_df['user_id'].nunique()}")


In [ ]:
# 4. CHIA TẬP THEO USER_ID (GroupShuffleSplit: 70% Train / 15% Val / 15% Test)
# Đảm bảo 100% không trùng user giữa Train, Val và Test -> không rò rỉ hành vi tương lai!
gss_outer = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss_outer.split(full_df, full_df['is_correct'], groups=full_df['user_id']))

train_df = full_df.iloc[train_idx].copy()
temp_df = full_df.iloc[temp_idx].copy()

gss_inner = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_rel_idx, test_rel_idx = next(gss_inner.split(temp_df, temp_df['is_correct'], groups=temp_df['user_id']))

val_df = temp_df.iloc[val_rel_idx].copy()
test_df = temp_df.iloc[test_rel_idx].copy()

# Kiểm tra chắc chắn không trùng user_id
assert set(train_df['user_id']).isdisjoint(set(val_df['user_id']))
assert set(train_df['user_id']).isdisjoint(set(test_df['user_id']))
assert set(val_df['user_id']).isdisjoint(set(test_df['user_id']))
print(f"✅ Split chuẩn không trùng User — Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

# 5. TÍNH question_difficulty = 1 - mean(is_correct) CHỈ TRÊN TẬP TRAIN (CHỐNG TARGET LEAKAGE)
# Giá trị càng cao (gần 1.0) = câu hỏi càng KHÓ; càng thấp (gần 0.0) = câu hỏi càng DỄ
global_train_difficulty = float(1.0 - train_df['is_correct'].mean())
q_stats = train_df.groupby(q_col)['is_correct'].agg(['mean', 'count'])
# Làm mượt Bayesian (m-estimate smoothing, m=10) để tránh nhiễu ở các câu hỏi ít lượt làm
m_smooth = 10.0
q_pass_rate_smooth = (q_stats['mean'] * q_stats['count'] + (1.0 - global_train_difficulty) * m_smooth) / (q_stats['count'] + m_smooth)
q_diff_map = (1.0 - q_pass_rate_smooth).to_dict()

# Tính trung vị elapsed_time chỉ trên tập Train để điền khuyết
train_median_elapsed = float(train_df['elapsed_time'].median())
print(f"📌 Global Train Difficulty: {global_train_difficulty:.4f} | Median Elapsed Time: {train_median_elapsed:.1f} ms")

for split_df in [train_df, val_df, test_df]:
    split_df['question_difficulty'] = split_df[q_col].map(q_diff_map).fillna(global_train_difficulty)
    split_df['elapsed_time'] = split_df['elapsed_time'].fillna(train_median_elapsed)
    split_df['user_avg_elapsed_time'] = split_df['user_avg_elapsed_time'].fillna(train_median_elapsed)


In [ ]:
# 6. HUẤN LUYỆN MÔ HÌNH 1: POST-QUIZ FULL MODEL (7 ĐẶC TRƯNG)
FEATURES_FULL = [
    'user_history_acc',
    'user_questions_attempted',
    'question_difficulty',
    'rolling_acc_5',
    'elapsed_time',
    'user_avg_elapsed_time',
    'log_time_diff'
]

X_train, y_train = train_df[FEATURES_FULL], train_df['is_correct']
X_val, y_val = val_df[FEATURES_FULL], val_df['is_correct']
X_test, y_test = test_df[FEATURES_FULL], test_df['is_correct']

class_ratio = float((y_train == 0).sum()) / float((y_train == 1).sum())

model_full = xgb.XGBClassifier(
    n_estimators=1200,
    learning_rate=0.02,
    max_depth=6,
    min_child_weight=4,
    subsample=0.85,
    colsample_bytree=0.8,
    scale_pos_weight=class_ratio,
    gamma=0.5,
    random_state=42,
    eval_metric="auc",
    early_stopping_rounds=80
)

print("⏳ Đang huấn luyện Mô hình 1 (Post-Quiz 7 biến)...")
model_full.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=100
)

y_pred_proba = model_full.predict_proba(X_test)[:, 1]
auc_full = roc_auc_score(y_test, y_pred_proba)
acc_full = accuracy_score(y_test, (y_pred_proba >= 0.5).astype(int))
ll_full = log_loss(y_test, y_pred_proba)

print(f"🚀 [Post-Quiz Model] Test ROC-AUC: {auc_full:.4f} | Accuracy: {acc_full:.4f} | LogLoss: {ll_full:.4f}")
model_full.save_model('/content/drive/MyDrive/DoAn4/ednet_xgboost_model_final.json')
print("💾 Đã lưu 'ednet_xgboost_model_final.json'")


In [ ]:
# 7. HUẤN LUYỆN MÔ HÌNH 2: PRE-QUIZ MODEL (6 ĐẶC TRƯNG - KHÔNG DÙNG elapsed_time)
# Dùng khi dự đoán độ khó trước khi học viên làm câu hỏi -> Tránh hoàn toàn lỗi OOD (elapsed_time = 0)
FEATURES_PREQUIZ = [
    'user_history_acc',
    'user_questions_attempted',
    'question_difficulty',
    'rolling_acc_5',
    'user_avg_elapsed_time',
    'log_time_diff'
]

model_prequiz = xgb.XGBClassifier(
    n_estimators=1200,
    learning_rate=0.02,
    max_depth=6,
    min_child_weight=4,
    subsample=0.85,
    colsample_bytree=0.8,
    scale_pos_weight=class_ratio,
    gamma=0.5,
    random_state=42,
    eval_metric="auc",
    early_stopping_rounds=80
)

print("⏳ Đang huấn luyện Mô hình 2 (Pre-Quiz 6 biến)...")
model_prequiz.fit(
    train_df[FEATURES_PREQUIZ], y_train,
    eval_set=[(train_df[FEATURES_PREQUIZ], y_train), (val_df[FEATURES_PREQUIZ], y_val)],
    verbose=100
)

y_pred_pre = model_prequiz.predict_proba(test_df[FEATURES_PREQUIZ])[:, 1]
auc_pre = roc_auc_score(y_test, y_pred_pre)
print(f"🚀 [Pre-Quiz Model] Test ROC-AUC: {auc_pre:.4f}")
model_prequiz.save_model('/content/drive/MyDrive/DoAn4/ednet_xgboost_prequiz_model.json')
print("💾 Đã lưu 'ednet_xgboost_prequiz_model.json'")


In [ ]:
# 8. ĐÁNH GIÁ & TRỰC QUAN HÓA SHAP (GLOBAL SUMMARY + WATERFALL + ROC CURVE)
feature_vi = {
    'user_history_acc': 'Tỉ lệ đúng lịch sử',
    'user_questions_attempted': 'Tổng số câu đã làm',
    'question_difficulty': 'Độ khó câu hỏi (0: Dễ -> 1: Khó)',
    'rolling_acc_5': 'Phong độ 5 câu gần nhất',
    'elapsed_time': 'Thời gian suy nghĩ câu này (ms)',
    'user_avg_elapsed_time': 'Tốc độ suy nghĩ TB (ms)',
    'log_time_diff': 'Khoảng cách giữa các phiên (log)'
}

explainer = shap.TreeExplainer(model_full)
X_sample = X_test.sample(n=min(1500, len(X_test)), random_state=42)
X_sample_vi = X_sample.rename(columns=feature_vi)
shap_exp = explainer(X_sample)
shap_exp.feature_names = list(X_sample_vi.columns)

# Biểu đồ 1: SHAP Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_exp.values, X_sample_vi, show=False)
plt.title("SHAP Summary Plot (Tập Test - Không rò rỉ dữ liệu)")
plt.tight_layout()
plt.show()

# Biểu đồ 2: SHAP Waterfall cho 1 mẫu học viên
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_exp[0], show=False)
plt.title("SHAP Waterfall — Giải thích cục bộ cho 1 lượt làm bài")
plt.tight_layout()
plt.show()

# Biểu đồ 3: So sánh ROC Curve của 2 mô hình
fpr_f, tpr_f, _ = roc_curve(y_test, y_pred_proba)
fpr_p, tpr_p, _ = roc_curve(y_test, y_pred_pre)
plt.figure(figsize=(8, 6))
plt.plot(fpr_f, tpr_f, color='darkorange', lw=2, label=f'Post-Quiz 7 biến (AUC = {auc_full:.4f})')
plt.plot(fpr_p, tpr_p, color='teal', lw=2, label=f'Pre-Quiz 6 biến (AUC = {auc_pre:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('So sánh đường cong ROC trên tập Test (User-Grouped Split)')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
